<a href="https://colab.research.google.com/github/pbcquoc/vietocr/blob/master/vietocr_gettingstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Introduction
<p align="center">
<img src="https://raw.githubusercontent.com/pbcquoc/vietocr/master/image/vietocr.jpg" width="512" height="512">
</p>
This notebook describe how you can use VietOcr to train OCR model




In [ ]:
! pip install --quiet vietocr==0.3.4

# Train model



1.   Load your config
2.   Train model using your dataset above



Load the default config, we adopt VGG for image feature extraction

In [ ]:
from vietocr.tool.config import Cfg
from vietocr.model.trainer import Trainer

# Change the config 

* *data_root*: the folder save your all images
* *train_annotation*: path to train annotation
* *valid_annotation*: path to valid annotation
* *print_every*: show train loss at every n steps
* *valid_every*: show validation loss at every n steps
* *epochs*: number of epochs to train your model
* *export*: export weights to folder that you can use for inference
* *metrics*: number of sample in validation annotation you use for computing full_sequence_accuracy, for large dataset it will take too long, then you can reuduce this number



In [ ]:
config = Cfg.load_config_from_name('vgg_seq2seq')

### Set các tham số cần thiết để huấn luyện mô hình

In [ ]:
vocab = ''.join(['(', ')', '/', '0123456789'] + [chr(i) for i in range(int("0x1000", 16), int("0x109F", 16) + 1)]) 

In [ ]:
train_params = {
          'visualize_every':200,
         'print_every':200,
         'valid_every':5000,
          'iters':200000,
          'checkpoint':'./checkpoint/transformerocr_checkpoint.pth',
          'export':'./weights/seq2seqocr.pth',
          'metrics': 10000,
          'batch_size':32
         }

config['trainer'].update(train_params)

dataset_params = {
    'name': 'id_name_address_old_pink',
    'data_root':'/data2/sftp/vmg/upload/nic/du_lieu_ban_giao/nic/data/ocr_textline',
     'train_annotation':'train_annotation.txt',
     'valid_annotation':'test_annotation.txt',}

config['dataset'].update(dataset_params)

config['vocab'] = vocab
config['dataloader'] = {'num_workers':3, 'pin_memory':False}
config['predictor']['beamsearch'] = False
config['device'] = 'cuda:1'

In [ ]:
from pprint import pprint
pprint(config)

You should train model from our pretrained 

In [ ]:
trainer = Trainer(config, pretrained=True)

In [ ]:
trainer.load_weights('/data2/sftp/vmg/upload/nic/model/nic/ocr/model.pth')

### Visualize train dataset

In [ ]:
trainer.visualize_dataset(fontname='Padauk')

In [ ]:
trainer.train()

In [ ]:
trainer.precision()

### Lưu config để predict 

In [ ]:
cfg.save('config.yml')